# **Traditional ML Model Performance Summary Across All Targets**

This notebook aggregates and summarizes the predictive performance of traditional machine learning (TradML) models developed for multiple biological targets in the drug discovery project. The goal is to compute and compare model performance metrics for both training and test datasets using previously generated prediction files.

The workflow performs the following steps:


1. Locate prediction files
The notebook searches the project directory for CSV files containing model predictions generated earlier in the pipeline. These files follow the naming patterns:

 - *_train_preds_tradML.csv

 - *_test_preds_tradML.csv

2. Match train and test predictions for each target
Targets are automatically identified from the file names (e.g., bace1_train_preds_tradML.csv).

3. Extract true and predicted values
For each dataset:

- The true activity values (pIC50) are extracted.

- All prediction columns beginning with pred_ are identified as model outputs.

4. Compute evaluation metrics
For each model and target, the following metrics are calculated:

- R² , RMSE, N (number of valid observations used in the calculation)

- Metrics are computed separately for Training data and Test data

5. Compile results into a unified summary table


6. Save results for downstream analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os, re, glob
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/trad-ml-predictions"
OUT_CSV  = os.path.join(BASE_DIR, "tradML_train_test_metrics_all_targets.csv")

In [ ]:


def metrics(y_true, y_pred):
    mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
    if mask.sum() == 0:
        return np.nan, np.nan, 0
    yt = y_true[mask].astype(float)
    yp = y_pred[mask].astype(float)
    r2 = float(r2_score(yt, yp))
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    return r2, rmse, int(mask.sum())

# Find all train/test files
train_files = glob.glob(os.path.join(BASE_DIR, "*_train_preds_tradML.csv"))
test_files  = glob.glob(os.path.join(BASE_DIR, "*_test_preds_tradML.csv"))

# Map target -> paths
def target_from_name(path):
    name = os.path.basename(path)
    # e.g., 5-HT6_train_preds_tradML.csv -> 5-HT6
    return re.sub(r"_(train|test)_preds_tradML\.csv$", "", name)

train_map = {target_from_name(p): p for p in train_files}
test_map  = {target_from_name(p): p for p in test_files}

targets = sorted(set(train_map) & set(test_map))
print("Targets found:", targets)

rows = []

for tgt in targets:
    df_tr = pd.read_csv(train_map[tgt])
    df_te = pd.read_csv(test_map[tgt])

    if "pIC50" not in df_tr.columns or "pIC50" not in df_te.columns:
        print(f"[SKIP] {tgt}: missing pIC50 column.")
        continue

    ytr = df_tr["pIC50"]
    yte = df_te["pIC50"]

    # all prediction columns (pred_*)
    pred_cols = [c for c in df_tr.columns if c.startswith("pred_")]
    # keep only those also in test
    pred_cols = [c for c in pred_cols if c in df_te.columns]

    if not pred_cols:
        print(f"[SKIP] {tgt}: no pred_* columns found.")
        continue

    for col in pred_cols:
        r2_tr, rmse_tr, n_tr = metrics(ytr, df_tr[col])
        r2_te, rmse_te, n_te = metrics(yte, df_te[col])

        rows.append({
            "Target": tgt,
            "Model": col.replace("pred_", ""),
            "Train_N": n_tr,
            "Train_R2": r2_tr,
            "Train_RMSE": rmse_tr,
            "Test_N": n_te,
            "Test_R2": r2_te,
            "Test_RMSE": rmse_te,
        })

summary = pd.DataFrame(rows)

# Optional: sort nicely
summary = summary.sort_values(["Target", "Test_R2"], ascending=[True, False]).reset_index(drop=True)

summary.to_csv(OUT_CSV, index=False)
print("Saved summary →", OUT_CSV)

# Display top rows
summary.head(20)


Targets found: ['5-HT6', 'ache', 'bace1', 'buche', 'esr1', 'gsk-3beta', 'mao-b']
Saved summary → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/trad-ml-predictions/tradML_train_test_metrics_all_targets.csv


,Target,Model,Train_N,Train_R2,Train_RMSE,Test_N,Test_R2,Test_RMSE
0,5-HT6,SVR,752,0.883421,0.365303,187,0.619082,0.612431
1,5-HT6,GradientBoosting,752,0.878098,0.373550,187,0.614436,0.616155
2,5-HT6,KNeighborsRegressor,752,0.710820,0.575343,187,0.555341,0.661691
3,5-HT6,RandomForest,752,0.919567,0.303431,187,0.553051,0.663393
4,5-HT6,XGBoost,752,0.940454,0.261078,187,0.549371,0.666118
5,5-HT6,MLPRegressor,752,0.595961,0.680072,187,0.488730,0.709524
6,ache,SVR,5274,0.888663,0.428233,1173,0.681810,0.738922
7,ache,XGBoost,5274,0.913861,0.376669,1173,0.675487,0.746228
8,ache,GradientBoosting,5274,0.918134,0.367208,1173,0.658288,0.765748
9,ache,RandomForest,5274,0.906889,0.391615,1173,0.655065,0.769351
